# Q3. Monthly Net PO value and PO count

Scope (from `eda/scope_eda`): purchasing org **US00**, purchase orders only (category F, deleted items excluded), Document Date **Oct 2025 – Sep 2026**, USD. Measure = **Net PO value** (ordered value, not invoiced or paid spend).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # src/, for sap_data and charts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sap_data import load_scope, product_family, FAMILY_CATEGORY
from charts import SERIES, MUTED, INK_2, usd, pct, style, save

style()
pd.set_option("display.max_columns", 30)
PERIOD = "Oct 2025 – Sep 2026"

w = load_scope()
w["supplier"] = w["Name 1"] + " (" + w["Supplier"].fillna("n/a") + ")"
TOTAL = w["Net Order Value"].sum()
print(f"{len(w)} PO items | {w['Purchasing Document'].nunique()} POs | {TOTAL:,.0f} USD")

**Question.** How have monthly Net PO value and PO count (distinct PO numbers) evolved over the last 12
complete months, and are there seasonal spikes?

Months without any PO are kept as 0, so gaps stay visible.

In [ ]:
months = pd.period_range("2025-10", "2026-09", freq="M")
mon = (w.groupby("month")
         .agg(net=("Net Order Value", "sum"), po_count=("Purchasing Document", "nunique"), items=("Item", "size"))
         .set_axis(pd.PeriodIndex(w.groupby("month").size().index, freq="M"))
         .reindex(months, fill_value=0))
mon["avg_per_po"] = (mon["net"] / mon["po_count"]).where(mon["po_count"] > 0)
mon["net_mom"] = mon["net"].pct_change().replace([np.inf, -np.inf], np.nan)
mon.style.format({"net": "{:,.0f}", "avg_per_po": "{:,.0f}", "net_mom": "{:+.0%}"}, na_rep="–")

In [ ]:
active = mon[mon["po_count"] > 0]
stats = pd.DataFrame({
    "all 12 months": [mon.net.mean(), mon.net.median(), mon.net.std() / mon.net.mean(), (mon.po_count == 0).sum()],
    f"{len(active)} active months": [active.net.mean(), active.net.median(), active.net.std() / active.net.mean(), 0],
}, index=["mean net / month", "median net / month", "coefficient of variation", "months with no PO"])
stats.style.format("{:,.2f}")

In [ ]:
labels = [m.strftime("%b\n%Y") if m.month in (1, 10) else m.strftime("%b") for m in months]
x = np.arange(len(months))
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"hspace": 0.35})
a1.bar(x, mon["net"], color=SERIES[0], width=0.7)
a1.axhline(mon["net"].mean(), color=INK_2, lw=1, ls="--")
a1.text(len(x) - 0.5, mon["net"].mean(), "12-month mean", ha="right", va="bottom", color=INK_2, fontsize=9)
for i, v in enumerate(mon["net"]):
    if v > 0:
        a1.text(i, v, f"${v / 1e3:,.0f}k" if v >= 1e4 else f"${v:,.0f}", ha="center", va="bottom", color=INK_2, fontsize=8.5)
a1.yaxis.set_major_formatter(usd); a1.set_title("Net PO value per month")
a2.bar(x, mon["po_count"], color=SERIES[0], width=0.7)
for i, v in enumerate(mon["po_count"]):
    if v > 0:
        a2.text(i, v, f"{v}", ha="center", va="bottom", color=INK_2, fontsize=8.5)
a2.set_title("PO count per month (distinct PO numbers)")
for ax in (a1, a2):
    ax.grid(axis="x", visible=False)
    for i in np.flatnonzero(mon["po_count"].to_numpy() == 0):
        ax.text(i, 0, "no POs", rotation=90, ha="center", va="bottom", color=INK_2, fontsize=8)
a2.set_xticks(x, labels)
fig.suptitle(f"Monthly purchasing activity, US00, {PERIOD}", x=0.01, ha="left", fontweight="bold")
save(fig, "q3_monthly")

## Findings for Antonio

**Finding.** Purchasing is very uneven: **5 of 12 months have no POs at all** (Jan, Feb, Jun–Aug 2026). Activity peaks in **Oct 2025** ($867k, 336 POs) and **Apr–May 2026** ($577k and $1.09M). The coefficient of variation is **1.51** over all 12 months (0.96 over the 7 active months). The mean of $255k per month is far above the median of $5k. The average value per PO rose from ~$2.5k in autumn 2025 to ~$8.8k in May 2026, so the May peak comes from larger POs, not more POs.

**Decision it supports.** If this pattern were real, demand would be lumpy. Antonio could then bundle peak-period demand into framework contracts and agree capacity with key suppliers ahead of the peaks.

**Limitation / extra data.** One year of data can't prove *seasonality*. The peaks match university teaching terms in the shared training client, so we describe them as activity peaks, not seasonal demand. Real seasonality would need 2–3 years of history and sales/production plans.